In [1]:
from kafka import KafkaConsumer
import json
import matplotlib.pyplot as plt
import time
import os

In [2]:
# Initialize data storage for plotting
timestamps = []
predictions = []
conf_no_aeration = []
conf_aeration = []
SAVE_PATH = "/opt/local/water_quality_plot.png"
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

In [3]:
# Kafka configuration
def initialize_consumer():
    kafka_topic = "prediction_topic"
    kafka_bootstrap_servers = ["localhost:9092"]

    # Create Kafka consumer
    consumer = KafkaConsumer(
        kafka_topic,
        bootstrap_servers=kafka_bootstrap_servers,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='latest',
        enable_auto_commit=True
    )
    return consumer

In [4]:
# Receive all published messages and update plot
def update_plot(consumer):
    try:
        for message in consumer:
            # Parse the message
            sensor_data = message.value
            print(f"Received: {sensor_data}")

            # Update data storage
            timestamps.append(sensor_data['timestamp'])
            predictions.append(sensor_data['prediction'])
            conf_no_aeration.append(sensor_data['proba_no_areation'])
            conf_aeration.append(sensor_data['proba_areation'])

            # Keep only the last 100 entries for plotting
            if len(timestamps) > 100:
                timestamps.pop(0)
                predictions.pop(0)
                conf_no_aeration.pop(0)
                conf_aeration.pop(0)

            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

            # --- 3.3.1: Estado de aireación ---
            ax1.step(timestamps, predictions, where='post', color='blue', linewidth=2)
            ax1.set_title("Estado de Aireación por Timestamp (1=Sí, 0=No)")
            ax1.set_ylabel("Estado")
            ax1.grid(True, alpha=0.3)

            # --- 3.3.2: Confianza Clase Aireación (cuando es escogida) ---
            ax2.scatter(timestamps, conf_aeration, color='green', label='Confianza Aireación')
            ax2.set_title("Confianza: Clase Aireación (Solo cuando es la clase predicha)")
            ax2.set_ylabel("Probabilidad")
            ax2.set_ylim(0.5, 1.0) # La confianza siempre será > 0.5 si fue la elegida
            ax2.grid(True, alpha=0.3)

            # --- 3.3.3: Confianza Clase No Aireación (cuando es escogida) ---
            ax3.scatter(timestamps, conf_no_aeration, color='red', label='Confianza No Aireación')
            ax3.set_title("Confianza: Clase No Aireación (Solo cuando es la clase predicha)")
            ax3.set_ylabel("Probabilidad")
            ax3.set_xlabel("Timestamp")
            ax3.set_ylim(0.5, 1.0)
            ax3.grid(True, alpha=0.3)

            # Formatear el eje X para que no se amontonen los timestamps
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()

            # Save the plot as an image
            plt.savefig(SAVE_PATH)
            plt.close()

            break  # Process one message at a time
    except KeyboardInterrupt:
        print("Stopped consuming messages.")
        consumer.close()

# 👂 Escuchamos

In [ ]:
consumer = initialize_consumer()
print("Subscribed to Kafka topic 'water_quality'.")

try:
    while True:
        update_plot(consumer)
        
except KeyboardInterrupt:
    print("Stopped visualization.")
    consumer.close()

Subscribed to Kafka topic 'water_quality'.
Received: {'timestamp': 1773321134, 'prediction': 1, 'proba_no_areation': 0.0, 'proba_areation': 1.0}
Received: {'timestamp': 1773321135, 'prediction': 0, 'proba_no_areation': 1.0, 'proba_areation': 0.0}
Received: {'timestamp': 1773321136, 'prediction': 1, 'proba_no_areation': 0.0, 'proba_areation': 1.0}
Received: {'timestamp': 1773321137, 'prediction': 0, 'proba_no_areation': 1.0, 'proba_areation': 0.0}
Received: {'timestamp': 1773321138, 'prediction': 0, 'proba_no_areation': 1.0, 'proba_areation': 0.0}
Received: {'timestamp': 1773321139, 'prediction': 0, 'proba_no_areation': 0.9, 'proba_areation': 0.1}
Received: {'timestamp': 1773321140, 'prediction': 0, 'proba_no_areation': 0.9, 'proba_areation': 0.1}
Received: {'timestamp': 1773321141, 'prediction': 1, 'proba_no_areation': 0.2, 'proba_areation': 0.8}
Received: {'timestamp': 1773321142, 'prediction': 1, 'proba_no_areation': 0.0, 'proba_areation': 1.0}
Received: {'timestamp': 1773321143, 'pr